In [ ]:
# Core libraries for cleaning and exploring data
import pandas as pd
import numpy as np 
import pyarrow
# Read the CSV file into a DataFrame.
df_raw =pd.read_csv(r"C:\Users\Kimo Store\Documents\ENSO\Production_Crops_Livestock_E_All_Data_(Normalized).csv")


C:\Users\Kimo Store\AppData\Local\Temp\ipykernel_22096\1850456875.py:6: DtypeWarning: Columns (13: Note) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw =pd.read_csv(r"C:\Users\Kimo Store\Documents\ENSO\Production_Crops_Livestock_E_All_Data_(Normalized).csv")


  ### loading the data
- The file is a **CSV**, so `pd.read_csv` is the right reader (`read_excel` failed earlier because the format was wrong).
- It loads **4,209,110 rows x 14 columns** of FAOSTAT crops and livestock data in *normalized (long) format*: one row = one area + item + element + year.
- The `DtypeWarning` says `Note` has mixed types. It is harmless here, but `low_memory=False` or an explicit `dtype` would silence it.

In [ ]:
#copy the data frame to a new variable to avoid changing the original data frame
df = df_raw.copy()
df.head()

,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1961,1961,ha,0.0,A,NaN
1,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1962,1962,ha,0.0,A,NaN
2,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1963,1963,ha,0.0,A,NaN
3,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1964,1964,ha,0.0,A,NaN
4,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1965,1965,ha,0.0,A,NaN


### first look at the rows
- Each row holds a single value for **Area x Item x Element x Year**.
- The first rows (Afghanistan, almonds, area harvested, 1961 to 1965) show `0.0` with flag `A`, so a zero is an official figure and not a missing value. Missing values are flagged separately (`M`).
- Codes such as `'004` and `'01371` start with an apostrophe, which is only a formatting artifact of the export.

| العمود | معناه |
|---|---|
| Area / Area Code / Area Code (M49) | الدولة أو المنطقة وأكوادها |
| Item / Item Code / Item Code (CPC) | المحصول أو المنتج  |
| Element / Element Code | نوع القياس (المساحة المحصودة، الإنتاج، الإنتاجية...) |
| Year / Year Code | السنة (مكررة في العمودين) |
| Unit | الوحدة (`ha` = هكتار) |
| Value | القيمة الفعلية |
| Flag | مصدر القيمة: `A` رسمية، `E` مقدّرة، `I` محسوبة... |
| Note | ملاحظات |

In [ ]:
# is Year Code equal to Year in all rows?
print((df_raw["Year Code"] == df_raw["Year"]).all())

# is the note column empty?
print(df_raw["Note"].notna().sum())

True
83290


| Check | Result | Meaning |
|---|---|---|
| Is `Year Code` identical to `Year`? | `True` | Fully duplicated, so it is safe to drop |
| How many rows have a `Note`? | `83290` | Not empty, so inspect it before dropping |

**Decision**

| Column | Action |
|---|---|
| `Year Code` | Drop |
| `Note` | Keep for now, check its values first |

In [ ]:
print("Shape:", df.shape)
df.tail(3)

Shape: (4209110, 14)


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
4209107,5817,'902,Net Food Importing Developing Countries (NFIDCs),1735,'F1735,Vegetables Primary,5510,Production,2022,2022,t,1.018261e+08,A,NaN
4209108,5817,'902,Net Food Importing Developing Countries (NFIDCs),1735,'F1735,Vegetables Primary,5510,Production,2023,2023,t,1.043533e+08,A,NaN
4209109,5817,'902,Net Food Importing Developing Countries (NFIDCs),1735,'F1735,Vegetables Primary,5510,Production,2024,2024,t,1.074442e+08,E,NaN


### size and ordering of the file
- Shape confirmed: **(4,209,110, 14)**.
- The last rows belong to **regional aggregates** (here NFIDCs, Vegetables Primary), so the file lists countries first and aggregates at the end.
- The final row (2024) has flag `E`, so recent values are often estimates.

In [ ]:
# Column names
print(f"Column names: {df.columns.tolist()}")
Categorical_columns = []

Column names: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (CPC)', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']


### column names
- 14 columns: area, item, element, year, unit, value, flag and note, each with its code variants.
- `Categorical_columns = []` is an empty placeholder that is not used later.

In [ ]:
df_raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 4209110 entries, 0 to 4209109
Data columns (total 14 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Area Code        int64  
 1   Area Code (M49)  str    
 2   Area             str    
 3   Item Code        int64  
 4   Item Code (CPC)  str    
 5   Item             str    
 6   Element Code     int64  
 7   Element          str    
 8   Year Code        int64  
 9   Year             int64  
 10  Unit             str    
 11  Value            float64
 12  Flag             str    
 13  Note             str    
dtypes: float64(1), int64(5), str(8)
memory usage: 686.9 MB


### data types and memory
- 5 `int64` columns, 1 `float64` (`Value`) and 8 text columns.
- Memory is about **686.9 MB**.
- This output has no non-null counts, so the next cells compute them explicitly.

In [ ]:
# Null counts per column
print(df_raw.isna().sum())

# Exact memory per column, in MB
print((df_raw.memory_usage(deep=True) / 1024**2).round(1))

# Total memory in MB
print(round(df_raw.memory_usage(deep=True).sum() / 1024**2, 1))

Area Code                0
Area Code (M49)          0
Area                     0
Item Code                0
Item Code (CPC)          0
Item                     0
Element Code             0
Element                  0
Year Code                0
Year                     0
Unit                     0
Value                94355
Flag                     0
Note               4125820
dtype: int64
Index                0.0
Area Code           32.1
Area Code (M49)     48.2
Area                80.0
Item Code           32.1
Item Code (CPC)     58.6
Item               115.9
Element Code        32.1
Element             79.1
Year Code           32.1
Year                32.1
Unit                42.4
Value               32.1
Flag                36.1
Note                34.0
dtype: float64
686.9


### missing values and memory per column
- Only two columns have nulls: **`Value` (94,355 missing, about 2.2%)** and **`Note` (4,125,820 missing, about 98%)**.
- Memory is dominated by the repeated text columns: `Item` (115.9 MB), `Area` (80.0 MB), `Element` (79.1 MB) and `Item Code (CPC)` (58.6 MB).
- These repeated strings are the best candidates for the `category` dtype.

In [ ]:
df_raw.info(verbose=True, show_counts=True, memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 4209110 entries, 0 to 4209109
Data columns (total 14 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   Area Code        4209110 non-null  int64  
 1   Area Code (M49)  4209110 non-null  str    
 2   Area             4209110 non-null  str    
 3   Item Code        4209110 non-null  int64  
 4   Item Code (CPC)  4209110 non-null  str    
 5   Item             4209110 non-null  str    
 6   Element Code     4209110 non-null  int64  
 7   Element          4209110 non-null  str    
 8   Year Code        4209110 non-null  int64  
 9   Year             4209110 non-null  int64  
 10  Unit             4209110 non-null  str    
 11  Value            4114755 non-null  float64
 12  Flag             4209110 non-null  str    
 13  Note             83290 non-null    str    
dtypes: float64(1), int64(5), str(8)
memory usage: 686.9 MB


### full `info()` with non-null counts
- `Value` has **4,114,755** non-null entries and `Note` has **83,290**, which matches the null counts above.
- Every other column is fully populated.

In [ ]:
# 1. Repeated text -> category (biggest saving)
cat_cols = ["Area", "Area Code (M49)", "Item", "Item Code (CPC)",
            "Element", "Unit", "Flag", "Note"]
for col in cat_cols:
    df[col] = df[col].astype("category")

# 2. Downcast integers
df["Area Code"]    = df["Area Code"].astype("int16")
df["Item Code"]    = df["Item Code"].astype("int16")
df["Element Code"] = df["Element Code"].astype("int16")
df["Year"]         = df["Year"].astype("int16")
df["Year Code"]    = df["Year Code"].astype("int16")

# 3. Check the result
print(round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")

120.4 MB


### memory optimization
- Converting 8 repeated text columns to `category` and 5 integer columns to `int16` cut memory from **686.9 MB to 120.4 MB (about 82% less)**.
- This is a performance change only. No rows or values were removed.

In [ ]:
df.info(verbose=True, show_counts=True, memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 4209110 entries, 0 to 4209109
Data columns (total 14 columns):
 #   Column           Non-Null Count    Dtype   
---  ------           --------------    -----   
 0   Area Code        4209110 non-null  int16   
 1   Area Code (M49)  4209110 non-null  category
 2   Area             4209110 non-null  category
 3   Item Code        4209110 non-null  int16   
 4   Item Code (CPC)  4209110 non-null  category
 5   Item             4209110 non-null  category
 6   Element Code     4209110 non-null  int16   
 7   Element          4209110 non-null  category
 8   Year Code        4209110 non-null  int16   
 9   Year             4209110 non-null  int16   
 10  Unit             4209110 non-null  category
 11  Value            4114755 non-null  float64 
 12  Flag             4209110 non-null  category
 13  Note             83290 non-null    category
dtypes: category(8), float64(1), int16(5)
memory usage: 120.5 MB


### dtypes after optimization
- The result is 8 `category`, 5 `int16` and 1 `float64` columns.
- Row count and non-null counts are unchanged (`Value` 4,114,755, `Note` 83,290), so no data was lost.

In [ ]:
print(df[["Area Code", "Item Code", "Element Code", "Year"]].max())

Area Code        5817
Item Code       17530
Element Code     5513
Year             2024
dtype: int16


### do the codes fit in `int16`?
- Maximum values: Area Code 5,817, Item Code 17,530, Element Code 5,513 and Year 2024.
- All are below the `int16` limit of 32,767, so the downcast is safe.

In [ ]:
cols = ["Area Code", "Item Code", "Element Code", "Year", "Year Code"]
print((df[cols].astype("int64") == df_raw[cols]).all())

Area Code       True
Item Code       True
Element Code    True
Year            True
Year Code       True
dtype: bool


### the conversion is lossless
- Comparing the converted integer columns with the originals returns `True` for all five columns.
- Nothing overflowed or changed, so `df` can be used for the analysis.

In [ ]:
print(df.nunique())
print(df["Year"].min(), df["Year"].max())
print(df["Flag"].value_counts())
print(df["Note"].value_counts().head(20))

Area Code              244
Area Code (M49)        244
Area                   244
Item Code              301
Item Code (CPC)        301
Item                   301
Element Code            14
Element                  8
Year Code               64
Year                    64
Unit                    10
Value              1141494
Flag                     5
Note                     1
dtype: int64
1961 2024
Flag
A    1837346
E    1778734
I     387557
X     111118
M      94355
Name: count, dtype: int64
Note
Unofficial figure    83290
Name: count, dtype: int64


### coverage and data quality
**Coverage:** **244** areas, **301** items, **8** elements (14 element codes), **10** units and **64** years (**1961 to 2024**).

**Flags (data quality)**

| Flag | Rows | Meaning |
|---|---|---|
| A | 1,837,346 | Official figure |
| E | 1,778,734 | Estimated |
| I | 387,557 | Imputed |
| X | 111,118 | From an international organisation |
| M | 94,355 | Missing |

- Only about **44% of rows are official**, and about 42% are estimates, so results should be read with that in mind.
- The `M` count (94,355) equals the number of `NaN` values in `Value`, so every missing value is flagged.
- `Note` has a single value, **"Unofficial figure"** (83,290 rows). It adds a label but nothing beyond the flag.

In [ ]:
print(df.groupby(["Element Code", "Element", "Unit"], observed=True).size())

Element Code  Element                        Unit   
5111          Stocks                         An          117255
5112          Stocks                         1000 An      53845
5114          Stocks                         No            9194
5312          Area harvested                 ha          893484
5313          Laying                         1000 An      30823
5318          Milk Animals                   An           45801
5320          Producing Animals/Slaughtered  An          261580
5321          Producing Animals/Slaughtered  1000 An      51501
5412          Yield                          kg/ha       823058
5413          Yield                          No/An        16761
5417          Yield/Carcass Weight           kg/An       181253
5424          Yield/Carcass Weight           g/An         80944
5510          Production                     t          1625403
5513          Production                     1000 No      18208
dtype: int64


### elements and units
- Production (`5510`, tonnes) is the largest group with about 1.6 million rows, followed by area harvested (`5312`, ha) and yield (`5412`, kg/ha).
- The **same element appears with different units** (for example Stocks in `An`, `1000 An` and `No`; Production in `t` and `1000 No`).
- **Never sum or compare `Value` without filtering on `Element Code` or `Unit`**, otherwise tonnes get mixed with thousands of eggs or animals.

In [ ]:
print(df["Area"].unique().tolist())

['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belgium-Luxembourg', 'Belize', 'Benin', 'Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'China, Hong Kong SAR', 'China, Macao SAR', 'China, mainland', 'China, Taiwan Province of', 'Colombia', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', "Côte d'Ivoire", 'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Czechoslovakia', "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Ethiopia PDR', 'Faroe Islands', 'F

### `Area` mixes different kinds of entities
- **Countries** are the large majority of the list.
- **Regional and group aggregates** come last: World, continents and sub-regions, European Union (27), LDCs, LLDCs, SIDS, LIFDCs and NFIDCs. Mixing them with countries causes **double counting**.
- **Historical entities** cover limited years: USSR, Yugoslav SFR, Czechoslovakia, Ethiopia PDR, Sudan (former), Serbia and Montenegro, Belgium-Luxembourg.
- **China is split into five entries** (China, mainland, Hong Kong SAR, Macao SAR, Taiwan Province of). `China` is the aggregate of the others.
- Egypt appears as `Egypt`, and `Northern Africa` is available for regional comparison.

In [ ]:
crop_rows = df[
    df["Element Code"].isin([5312, 5412])
    & (df["Item Code"] < 1700)
    & df["Value"].notna()
]

crops_all = (
    crop_rows.groupby(["Item Code", "Item"])
    .agg(n_countries=("Area", "nunique"),
         first_year=("Year", "min"),
         last_year=("Year", "max"))
    .reset_index()
    .sort_values("Item Code")
)

print(len(crops_all), "crops in the dataset")
print(crops_all.to_string(index=False))


158 crops in the dataset
 Item Code                                                                      Item  n_countries  first_year  last_year
        15                                                                     Wheat          164        1961       2024
        27                                                                      Rice          179        1961       2024
        44                                                                    Barley          143        1961       2024
        56                                                              Maize (corn)          216        1961       2024
        71                                                                       Rye           96        1961       2024
        75                                                                      Oats          112        1961       2024
        79                                                                    Millet          124        1961       2024
       

### which crops can be analysed
- **158 primary crops** have area harvested or yield data. Item Code below 1700 excludes aggregates, and requiring area or yield excludes processed products such as oils.
- 154 of the 158 start in 1961 and 157 end in 2024. The exceptions are small crops: jojoba (from 1982), tallowtree seeds (from 1985), cassava leaves (1990 only) and one that starts in 2000.
- Coverage is uneven. Maize and tomatoes appear for 216 areas, potatoes for 201, rice for 179 and wheat for 164, while jojoba has 4, kapok 5 and quinoa 11. About 68 crops have 100 or more areas.
- `n_countries` counts distinct `Area` values, so it **includes regional aggregates**.
- For an ENSO analysis, prefer widely grown crops with the full 1961 to 2024 range, such as wheat, rice, maize, soya, sugar cane and cotton.

In [ ]:
# 1. Distribution of values (production in tonnes only, so units are comparable)
prod = df[df["Element Code"] == 5510]
print(prod["Value"].describe())
print("zeros:", (prod["Value"] == 0).sum(), " positive:", (prod["Value"] > 0).sum())

# 2. Year coverage per area (areas that do NOT cover all 64 years)

cov = df.groupby("Area", observed=True)["Year"].agg(first="min", last="max", years="nunique")
print((cov["years"] == 64).sum(), "areas cover all 64 years")
print(cov[cov["years"] < 64].sort_values("years").to_string())

# 3. Historical entities and their lifetimes

hist = ["USSR", "Yugoslav SFR", "Czechoslovakia", "Ethiopia PDR",
        "Sudan (former)", "Serbia and Montenegro", "Belgium-Luxembourg"]
print(cov.loc[hist])


# 4. Top 10 producers of wheat, rice and maize in 2023 (tonnes)

countries = df[(df["Area Code"] < 5000) & (df["Area"] != "China")]
for code, name in [(15, "Wheat"), (27, "Rice"), (56, "Maize (corn)")]:
    top = (countries[(countries["Item Code"] == code)
                     & (countries["Element Code"] == 5510)
                     & (countries["Year"] == 2023)]
           .nlargest(10, "Value")[["Area", "Value", "Flag"]])
    print("\\n", name); print(top.to_string(index=False))

# 5. Flag share by year: are recent figures more estimated?

flag_share = pd.crosstab(df["Year"], df["Flag"], normalize="index").round(3)
print(flag_share.iloc[::8])
print(flag_share.tail(3))

# 6. Livestock coverage

live_elements = ["Stocks", "Laying", "Milk Animals",
                 "Producing Animals/Slaughtered", "Yield/Carcass Weight"]
live = df[df["Element"].isin(live_elements)]
print(live.groupby(["Element", "Unit"], observed=True)ٍ
          .agg(items=("Item", "nunique"), areas=("Area", "nunique")))
print(sorted(df.loc[df["Element"] == "Stocks", "Item"ٍ].unique()))

count    1.580582e+06
mean     2.543851e+06
std      2.870742e+07
min      0.000000e+00
25%      1.656000e+03
50%      2.070000e+04
75%      2.086200e+05
max      3.133223e+09
Name: Value, dtype: float64
zeros: 45904  positive: 1534678
199 areas cover all 64 years
                                  first  last  years
Area                                                
South Sudan                        2012  2024     13
Sudan                              2012  2024     13
Serbia and Montenegro              1992  2005     14
Montenegro                         2006  2024     19
Serbia                             2006  2024     19
Belgium                            2000  2024     25
Luxembourg                         2000  2024     25
Yugoslav SFR                       1961  1991     31
USSR                               1961  1991     31
Slovakia                           1993  2024     32
Palestine                          1991  2022     32
Eritrea                            1993  2024 

### Conclusion: year coverage per area

- **199 of 244 areas cover all 64 years** (1961 to 2024). The other 45 have shorter series.
- Most short series come from **changes in country borders**:
  - Former Soviet states, former Yugoslav states and Czechoslovakia successors start in **1992 or 1993** (for example Russian Federation, Ukraine, Kazakhstan, Czechia, Slovakia, Croatia).
  - **Ethiopia** starts in 1993 (Ethiopia PDR covers 1961 to 1992) and **Eritrea** in 1993.
  - **Sudan** and **South Sudan** both start in 2012 (Sudan (former) covers 1961 to 2011).
  - **Belgium** and **Luxembourg** start in 2000 (Belgium-Luxembourg covers 1961 to 1999).
  - **French Guiana, Guadeloupe, Martinique and Réunion** end in 2006.
  - **Nicaragua** is missing only 2024, and **Palestine** covers 1991 to 2022.
- **Central Asia** (a regional aggregate) also starts in 1992.
- **Implication:** a long time series (such as an ENSO study over 1961 to 2024) is only possible for the 199 full-coverage areas. Successor states have at most about 33 years unless predecessor and successor series are stitched together.

### Conclusion: historical entities

| Entity | First year | Last year | Years |
|---|---|---|---|
| USSR | 1961 | 1991 | 31 |
| Yugoslav SFR | 1961 | 1991 | 31 |
| Czechoslovakia | 1961 | 1992 | 32 |
| Ethiopia PDR | 1961 | 1992 | 32 |
| Sudan (former) | 1961 | 2011 | 51 |
| Serbia and Montenegro | 1992 | 2005 | 14 |
| Belgium-Luxembourg | 1961 | 1999 | 39 |

- Each historical entity ends exactly when its successors begin, so there is **no overlap and no gap**, but also no continuous series.
- These entities must be kept separate from current countries, or the same land will be counted twice when areas are summed.

### Conclusion: top producers in 2023 (tonnes)

| Rank | Wheat | Rice | Maize (corn) |
|---|---|---|---|
| 1 | China, mainland (136.6 Mt) | India (207.8 Mt) | United States (389.7 Mt) |
| 2 | India (110.6 Mt) | China, mainland (206.6 Mt) | China, mainland (288.8 Mt) |
| 3 | Russian Federation (92.8 Mt, flag X) | Bangladesh (60.6 Mt) | Brazil (131.9 Mt) |
| 4 | United States (49.3 Mt) | Indonesia (54.0 Mt) | Argentina (41.4 Mt) |
| 5 | France (36.0 Mt) | Viet Nam (43.5 Mt) | India (36.7 Mt) |
| 6 | Canada (33.4 Mt) | Thailand (33.9 Mt) | Ukraine (31.0 Mt) |
| 7 | Pakistan (28.2 Mt) | Myanmar (25.65 Mt, flag X) | Mexico (27.5 Mt) |
| 8 | Australia (28.0 Mt) | Philippines (20.1 Mt) | South Africa (16.4 Mt) |
| 9 | Türkiye (22.0 Mt) | Pakistan (14.8 Mt) | Canada (15.4 Mt) |
| 10 | Ukraine (21.6 Mt) | Cambodia (13.2 Mt) | Russian Federation (15.0 Mt, flag X) |

- The rankings look plausible, which supports the data quality at the top of the distribution.
- Almost all values are official (`A`). The exceptions are Russia (wheat and maize) and Myanmar (rice), which are `X` (from an international organisation) and are round numbers.
- Egypt is not in the top 10 for wheat (about 9.1 Mt in 2023).
- Production is highly concentrated: in rice, India and China alone produce about 414 Mt, more than the other eight listed countries combined (about 291 Mt).

### Conclusion: flag share by year

| Year | A (official) | E (estimated) | I (imputed) | M (missing) | X (intl. org.) |
|---|---|---|---|---|---|
| 1961 | 35.8% | 53.7% | 5.7% | 3.0% | 1.8% |
| 1985 | 40.4% | 49.6% | 5.7% | 1.7% | 2.6% |
| 2001 | 45.5% | 38.8% | 9.9% | 2.3% | 3.4% |
| 2017 | 47.2% | 33.3% | 16.4% | 1.5% | 1.7% |
| 2023 | 42.6% | 38.7% | 15.4% | 1.0% | 2.3% |
| 2024 | 37.7% | 43.9% | 16.1% | 1.1% | 1.2% |

- The official share **rose from about 36% (1961) to a peak of about 47% (2009 to 2017)**, then **fell to about 38% in 2024**.
- The estimated share fell from about 54% to about 33% (2017) and rose again to about 44% in 2024.
- The imputed share grew steadily from about 6% to about 16%.
- **Implication:** 2023 and 2024 are the least reliable recent years, so they are likely to be revised. The earliest years (1961 to 1977) are also more than 45% estimated. Estimated and imputed values can be smoother than real measurements, which could weaken year-to-year relationships such as ENSO effects.
- These shares are over all rows in the file (all elements, units, and aggregate rows), not only individual crops.

### Conclusion: livestock coverage

| Element | Unit | Items | Areas |
|---|---|---|---|
| Stocks | An | 12 | 241 |
| Stocks | 1000 An | 8 | 240 |
| Stocks | No | 1 | 164 |
| Producing Animals/Slaughtered | An | 31 | 242 |
| Producing Animals/Slaughtered | 1000 An | 8 | 240 |
| Yield/Carcass Weight | kg/An | 24 | 242 |
| Yield/Carcass Weight | g/An | 11 | 240 |
| Milk Animals | An | 6 | 232 |
| Laying | 1000 An | 3 | 239 |

- Livestock is well covered: most measurements exist for about 230 to 242 areas.
- **Stocks** cover 21 items: asses, bees, buffalo, camels, cattle, chickens, ducks, geese, goats, horses, mules and hinnies, other birds, other camelids, other rodents, rabbits and hares, sheep, swine, turkeys, plus three aggregates (`Cattle and Buffaloes`, `Sheep and Goats`, `Poultry Birds`).
- The **same element uses different units for different items** (`An` versus `1000 An`), so values must never be compared or summed without filtering by unit. The single `No` unit item is likely bees (hives).
- Stocks aggregates must be excluded when summing across items to avoid double counting.